# 🏦 Financial Earnings Report Analysis Desk (FERAD)
## Multi-Agent System Demo Notebook
**Group 9 | Microsoft Semantic Kernel Architecture | Fan-Out / Fan-In Pattern**

---

This notebook demonstrates the full multi-agent pipeline trace for the FERAD system.
Each cell corresponds to a stage in the fan-out / fan-in architecture.

### Agents:
1. 📥 **Extraction Agent** — pulls KPIs from the report (regex + RAG)
2. 📈 **Revenue Analyst** — revenue & growth commentary
3. ⚠️ **Risk Analyst** — risk signals and red flags
4. 🔭 **Outlook Analyst** — forward guidance assessment
5. 📊 **Comparison Agent** — QoQ delta table (Group-of-3 add-on)
6. ✍️ **Editor Manager** — fan-in, disagreement detection, final note

In [183]:
# ── Setup: install required packages (run once) ──────────────────────────
# Uncomment the line below if packages are not yet installed:
# !pip install rich pdfplumber -q

import sys, os
sys.path.insert(0, os.getcwd())  # ensure local modules are importable

print('✅ Environment ready')

✅ Environment ready


## API Key & LLM Provider Configuration

FERAD supports **Groq**, **OpenAI**, **Azure OpenAI**, or **Offline Grounded Mode**.
- **Groq:** Ultra-fast open-weight LLM inference (`llama-3.3-70b-versatile`, `mixtral-8x7b-32768`).
- Keys are loaded automatically from `.env`, or can be pasted directly in the cell below.

In [184]:
# ── API Key & Model Provider Setup ────────────────────────
import os
from dotenv import load_dotenv, find_dotenv
import llm_client

# 1. Load keys automatically from .env (forced override)
load_dotenv(find_dotenv(), override=True)

# 2. Optional: Paste or override your GROQ_API_KEY directly here:
# os.environ["GROQ_API_KEY"] = "gsk_your_groq_api_key_here"

groq_key = os.getenv("GROQ_API_KEY", "").strip()

print("=" * 60)
print("  API KEY & MODEL PROVIDER STATUS")
print("=" * 60)
if groq_key:
    masked = groq_key[:6] + "..." + groq_key[-4:] if len(groq_key) > 10 else "***"
    print(f"🔑 Groq API Key       : Detected ({masked})")
    print(f"⚡ Model Deployment   : {os.getenv('GROQ_MODEL_NAME', 'qwen/qwen3.8-27b')}")
    print(f"🤖 Active Provider    : {llm_client.get_llm_provider()}")
else:
    print("🔑 Groq API Key       : Not set in .env (Offline mode active)")
    print("ℹ️  To enable Groq, set GROQ_API_KEY in .env or uncomment line 10 above.")
    print("⚡ Execution Mode     : Grounded Deterministic RAG (Offline, 0 API cost)")


  API KEY & MODEL PROVIDER STATUS
🔑 Groq API Key       : Detected (gsk_Um...GvnP)
⚡ Model Deployment   : qwen/qwen3.8-27b
🤖 Active Provider    : groq


## Step 0 – Initialise Shared Memory (Blackboard)

In [185]:
from shared_memory import SharedMemory
from document_loader import load_from_txt_file, make_previous_quarter_figures

# Create the central shared blackboard
memory = SharedMemory()

# Load the current quarter earnings report
memory.raw_text = load_from_txt_file('sample_report_q3_fy2024.txt')

print(f'📄 Report loaded: {len(memory.raw_text):,} characters')
print(f'📋 Preview (first 300 chars):')
print(memory.raw_text[:300])

📄 Report loaded: 7,098 characters
📋 Preview (first 300 chars):
APEX TECHNOLOGIES INC.
Q3 FY2024 EARNINGS REPORT
For the Quarter Ended September 30, 2024

FINANCIAL HIGHLIGHTS

Revenue and Profitability

Apex Technologies Inc. reported total revenues of $8.62 billion for the third quarter of fiscal year 2024, representing a year-over-year growth of 18.4% compare


## Step 1 – Extraction Agent [SEQUENTIAL]

**Architecture Note:** Extraction must run first as all downstream agents depend on `SharedMemory.current_figures`.

**Tools called:**
- `parse_financial_figures()` — regex-based KPI extraction
- `retrieve_relevant_chunks()` — keyword-scored RAG over document chunks

In [186]:
from extraction_agent import ExtractionAgent

print('=' * 60)
print('  EXTRACTION AGENT')
print('=' * 60)

ext_agent = ExtractionAgent(memory)
figures = ext_agent.run()

print(f'\n✅ Extracted {len(figures)} KPIs from the report:')
print()

# Display extracted figures as a table
print(f'{"Metric":<25} {"Value":<15} {"Unit":<10}')
print('-' * 55)
for key, obj in figures.items():
    val = obj.get('value', 'N/A')
    unit = obj.get('unit', '')
    if isinstance(val, float):
        val_str = f'{val:,.2f}'
    else:
        val_str = str(val)
    print(f'{key:<25} {val_str:<15} {unit:<10}')

print()
print('📍 Source tracing (first 3 KPIs):')
for key in list(figures.keys())[:3]:
    print(f'  [{key}] → "{figures[key]["source"][:100]}"')

  EXTRACTION AGENT

✅ Extracted 7 KPIs from the report:

Metric                    Value           Unit      
-------------------------------------------------------
revenue                   8,620.00        M USD     
operating_income          1,980.00        M USD     
gross_margin              63.90           %         
operating_margin          22.90           %         
yoy_growth                31.20           %         
guidance_revenue          9,100.00        M USD     
guidance_eps              3.40            USD       

📍 Source tracing (first 3 KPIs):
  [revenue] → "reported total revenues of $8.62 billion for the third quarter of fiscal year 2024, representing a y"
  [operating_income] → "Operating income was $1.98 billion for the quarter, resulting in an operating margin of 22.9%."
  [gross_margin] → "Gross profit for Q3 FY2024 was $5.51 billion, reflecting a gross margin of 63.9%, up from 61.2% in t"


In [187]:
# Load previous quarter for comparison
from document_loader import load_from_txt_file
from extraction_agent import ExtractionAgent
from shared_memory import SharedMemory as SM2

print('📊 Loading previous quarter (Q2 FY2024)...')
prev_mem = SM2()
prev_mem.raw_text = load_from_txt_file('sample_report_q2_fy2024.txt')
ExtractionAgent(prev_mem).run()
memory.previous_figures = prev_mem.current_figures

print(f'✅ Previous quarter figures loaded: {list(memory.previous_figures.keys())}')

📊 Loading previous quarter (Q2 FY2024)...
✅ Previous quarter figures loaded: ['revenue', 'operating_income', 'gross_margin', 'operating_margin', 'yoy_growth', 'guidance_revenue']


## Step 2 – FAN-OUT: Three Analyst Agents (Conceptually Parallel)

Each analyst reads from the shared blackboard independently.
In a production SK deployment these would run as concurrent kernel invocations.

In [188]:
from revenue_analyst import RevenueAnalyst

print('=' * 60)
print('  REVENUE ANALYST  [FAN-OUT Branch 1/3]')
print('=' * 60)

rev_agent = RevenueAnalyst(memory)
revenue_section = rev_agent.run()

print(revenue_section)

  REVENUE ANALYST  [FAN-OUT Branch 1/3]
## Revenue & Growth Analysis

**Headline Revenue:** $8,620.0 M USD
> *Source:* "reported total revenues of $8.62 billion for the third quarter of fiscal year 2024, representing a year-over-year growth of 18.4% compared to $7.28 bi"

**Year-over-Year Growth:** Revenue grew by **31.2%** YoY.
> *Source:* "Segment Performance

Cloud Services revenue grew 31.2% year-over-year to $4.18 billion, now representing 48.5% of total company revenue."

**Quarter-over-Quarter Growth:** ↑ growth of **12.2%** QoQ.

**Profitability Margins:**
  - Gross Margin: **63.9%**
    > *Source:* "Gross profit for Q3 FY2024 was $5.51 billion, reflecting a gross margin of 63.9%, up from 61.2% in the prior-year quarter."
  - Operating Margin: **22.9%**
    > *Source:* "Operating income was $1.98 billion for the quarter, resulting in an operating margin of 22.9%."

**Revenue Assessment:** 🟢 STRONG



In [189]:
from risk_analyst import RiskAnalyst

print('=' * 60)
print('  RISK ANALYST  [FAN-OUT Branch 2/3]')
print('=' * 60)

risk_agent = RiskAnalyst(memory)
risk_section = risk_agent.run()

print(risk_section)

  RISK ANALYST  [FAN-OUT Branch 2/3]
## Risk Analysis

**Identified Risk Signals:**

### Guidance Cut / Warning  [HIGH 🔴]
> *Source:* "Management Cautionary Remarks

Despite the positive outlook, management noted several areas of uncertainty."

### Margin Compression  [HIGH 🔴]
> *Source:* "Supply Chain and Operational Risks

The company continues to experience supply chain disruptions in its Hardware & Devices segment."

### Revenue / Demand Risk  [HIGH 🔴]
> *Source:* "Hardware & Devices revenue declined 4.8% to $1.50 billion due to component shortages and challenging macroeconomic conditions affecting consumer and enterprise hardware spending."

### Debt / Liquidity Risk  [MEDIUM 🟡]
> *Source:* "Debt and Capital Structure

The company holds $14.2 billion in long-term debt, with $2.8 billion maturing within the next 12 months."

### Legal / Regulatory Risk  [MEDIUM 🟡]
> *Source:* "Regulatory and Legal Matters

Apex Technologies is subject to an ongoing regulatory investigation by the E

In [190]:
from outlook_analyst import OutlookAnalyst

print('=' * 60)
print('  OUTLOOK ANALYST  [FAN-OUT Branch 3/3]')
print('=' * 60)

out_agent = OutlookAnalyst(memory)
outlook_section = out_agent.run()

print(outlook_section)

  OUTLOOK ANALYST  [FAN-OUT Branch 3/3]
## Outlook & Forward Guidance

**Quantitative Guidance Provided:**
  - Revenue Guidance: $9,100.0 M USD
    > *Source:* "FORWARD GUIDANCE AND OUTLOOK

Revenue Guidance

For Q4 FY2024, Apex Technologies expects total revenue in the range of $9.1 billion to $9.4 billion, i"
  - EPS Guidance: $3.40
    > *Source:* "EPS Guidance

For Q4 FY2024, the company expects diluted EPS in the range of $3.40 to $3.55, reflecting continued investment in AI and cloud infrastru"

**Guidance Specificity:** HIGH – Quantitative targets provided.

**Management Tone:** 🟢 BULLISH – 7 positive vs. 1 cautious signals.

**Key Forward-Looking Statements:**

1. "This compares to operating income of $1.54 billion and an operating margin of 21.2% in Q3 FY2023, reflecting disciplined expense management despite continued investment in research and development."
2. "Management does not expect the investigation to result in material financial liability, though the outcome remains 

In [191]:
from comparison_agent import ComparisonAgent

print('=' * 60)
print('  COMPARISON AGENT  [Group-of-3 Add-on]')
print('=' * 60)

comp_agent = ComparisonAgent(memory)
comparison_section = comp_agent.run()

print(comparison_section)

  COMPARISON AGENT  [Group-of-3 Add-on]
## Quarter-over-Quarter Comparison

| Metric | Prior Q | Current Q | Δ Absolute | Δ % | Dir |
|--------|---------|-----------|-----------|-----|-----|
| Gross Margin | 62.8% | 63.9% | 1.1% | 1.8% | ↑ |
| Guidance EPS | N/A | 3.4 | N/A | N/A | — |
| Guidance Revenue | $8,400.0M | $9,100.0M | $700.0M | 8.3% | ↑ |
| Operating Income | $1,610.0M | $1,980.0M | $370.0M | 23.0% | ↑ |
| Operating Margin | 21.0% | 22.9% | 1.9% | 9.0% | ↑ |
| Revenue | $7,680.0M | $8,620.0M | $940.0M | 12.2% | ↑ |
| YoY Growth | 26.5% | 31.2% | 4.7% | 17.7% | ↑ |

**Top Movers (QoQ):**
  - **Operating Income**: ↑ 23.0% — _Operating income was $1.98 billion for the quarter, resulting in an operating margin of 22.9%._
  - **YoY Growth**: ↑ 17.7% — _Segment Performance

Cloud Services revenue grew 31.2% year-over-year to $4.18 billion, now representing 48.5% of total company revenue._
  - **Revenue**: ↑ 12.2% — _reported total revenues of $8.62 billion for the third quarter o

## Step 3 – FAN-IN: Editor Manager

The Editor Manager:
1. Collects all three analyst sections from shared memory
2. Detects any disagreements between analyst sentiments
3. Assembles the final one-page analyst note

**Tools called:**
- `detect_disagreements()` — cross-analyst sentiment comparison
- `assemble_analyst_note()` — final note assembly with sourced figures

In [192]:
from editor_manager import EditorManager

print('=' * 60)
print('  EDITOR MANAGER  [FAN-IN]')
print('=' * 60)

editor = EditorManager(
    memory, 
    company_name='Apex Technologies Inc.',
    quarter_label='Q3 FY2024'
)
final_note = editor.run()

print('✅ Final analyst note assembled!')
print(f'   Length: {len(final_note):,} characters')
print(f'   Saved to: analyst_note.md')

# Save to file
with open('analyst_note.md', 'w', encoding='utf-8') as f:
    f.write(final_note)

print()
print(final_note)

  EDITOR MANAGER  [FAN-IN]
✅ Final analyst note assembled!
   Length: 6,358 characters
   Saved to: analyst_note.md

# ANALYST NOTE: APEX TECHNOLOGIES INC. – Q3 FY2024 Earnings
**Date:** 02 November 2026
**Classification:** For Educational Purposes | Multi-Agent Generated Report
**System:** Financial Earnings Report Analysis Desk (FERAD)

---

## Executive Summary

**Apex Technologies Inc.** reported revenue of **$8,620.0 M USD** for Q3 FY2024.
Revenue grew **31.2%** year-over-year.
Gross margin stood at **63.9%**.

---

## Revenue & Growth Analysis

**Headline Revenue:** $8,620.0 M USD
> *Source:* "reported total revenues of $8.62 billion for the third quarter of fiscal year 2024, representing a year-over-year growth of 18.4% compared to $7.28 bi"

**Year-over-Year Growth:** Revenue grew by **31.2%** YoY.
> *Source:* "Segment Performance

Cloud Services revenue grew 31.2% year-over-year to $4.18 billion, now representing 48.5% of total company revenue."

**Quarter-over-Quarter Growth:

### Optional: Groq LLM Executive Takeaways

If a `GROQ_API_KEY` is present, this cell calls Groq's high-speed inference engine (`llama-3.3-70b-versatile`) to generate a concise 3-bullet executive synthesis grounded in the extracted KPIs.

In [193]:
import os
import llm_client

groq_key = os.getenv("GROQ_API_KEY", "").strip()
if groq_key:
    print("🤖 Calling Groq LLM (llama-3.3-70b-versatile)...\n")
    kpis = memory.current_figures
    rev_val = kpis.get('revenue', {}).get('value', 'N/A')
    gm_val = kpis.get('gross_margin', {}).get('value', 'N/A')
    yoy_val = kpis.get('yoy_growth', {}).get('value', 'N/A')
    prompt = f"""Analyze the following earnings data for Apex Technologies Inc. (Q3 FY2024):
- Revenue: ${rev_val}M USD
- Gross Margin: {gm_val}%
- YoY Growth: {yoy_val}%
Provide exactly 3 concise, high-impact bullet points for an executive briefing."""
    try:
        briefing = llm_client.query_groq(prompt, system_prompt="You are a Wall Street equity research director.")
        print("=" * 60)
        print("  GROQ LLM SYNTHESIS (llama-3.3-70b-versatile)")
        print("=" * 60)
        print(briefing)
    except Exception as e:
        print(f"⚠️ Groq query notice: {e}")
else:
    print("ℹ️ Groq API key is not configured. Running in deterministic grounded offline mode.")
    print("   All claims and calculations in the analyst note are fully grounded via RAG and regex.")


🤖 Calling Groq LLM (llama-3.3-70b-versatile)...

  GROQ LLM SYNTHESIS (llama-3.3-70b-versatile)
*   **Robust Top-Line Acceleration:** Q3 revenue of $8.62B reflects a strong 31.2% YoY growth rate, signaling significant market share gains and sustained demand momentum that far exceeds typical sector averages.
*   **High-Quality Profitability:** The 63.9% gross margin underscores exceptional pricing power and operational efficiency, indicating that revenue growth is being captured with minimal dilution to unit economics.
*   **Scale-Driven Margin Expansion:** The combination of double-digit growth and high-margin retention suggests Apex is successfully leveraging scale to drive operating leverage, positioning the firm for continued EPS upside in subsequent quarters.


## Step 4 – Multi-Agent Trace (Blackboard Audit Log)

In [194]:
print('=' * 60)
print('  MULTI-AGENT TRACE  (Shared Memory Audit Log)')
print('=' * 60)
print()

# Print every recorded trace event
print(f'{"#":<4} {"Agent":<22} {"Action":<25} {"Data"}')
print('-' * 90)
for i, event in enumerate(memory.agent_trace, 1):
    data_str = str(event.get('data', ''))[:55]
    agent_str = event.get('agent', '')[:22]
    action_str = event.get('action', '')[:25]
    print(f'{i:<4} {agent_str:<22} {action_str:<25} {data_str}')

print()
print(f'Total trace events: {len(memory.agent_trace)}')
print()

# Agents seen in trace
agents_seen = set(e['agent'] for e in memory.agent_trace)
print(f'Agents in trace: {agents_seen}')

  MULTI-AGENT TRACE  (Shared Memory Audit Log)

#    Agent                  Action                    Data
------------------------------------------------------------------------------------------
1    Extraction Agent       START                     Beginning structured extraction + RAG
2    Extraction Agent       CALL_TOOL                 parse_financial_figures
3    Extraction Agent       TOOL_RESULT               7 figures extracted
4    Extraction Agent       CALL_TOOL                 retrieve_relevant_chunks
5    Extraction Agent       TOOL_RESULT               4 relevant chunks retrieved
6    Extraction Agent       WRITE_MEMORY              ['revenue', 'operating_income', 'gross_margin', 'operat
7    Extraction Agent       END                       Extraction complete – 7 KPIs
8    Revenue Analyst        START                     Reading KPIs from shared memory
9    Revenue Analyst        CALL_TOOL                 compute_revenue_metrics
10   Revenue Analyst        TOOL_RESULT 

## Step 5 – Shared Memory Snapshot

In [195]:
import json

print('=' * 60)
print('  SHARED MEMORY SNAPSHOT')
print('=' * 60)
snapshot = memory.snapshot()
print(json.dumps(snapshot, indent=2, default=str))

  SHARED MEMORY SNAPSHOT
{
  "raw_text_length": 7098,
  "current_figures": {
    "revenue": {
      "value": 8620.0,
      "unit": "M USD",
      "source": "reported total revenues of $8.62 billion for the third quarter of fiscal year 2024, representing a year-over-year growth of 18.4% compared to $7.28 billion in Q3 FY2023."
    },
    "operating_income": {
      "value": 1980.0,
      "unit": "M USD",
      "source": "Operating income was $1.98 billion for the quarter, resulting in an operating margin of 22.9%."
    },
    "gross_margin": {
      "value": 63.9,
      "unit": "%",
      "source": "Gross profit for Q3 FY2024 was $5.51 billion, reflecting a gross margin of 63.9%, up from 61.2% in the prior-year quarter."
    },
    "operating_margin": {
      "value": 22.9,
      "unit": "%",
      "source": "Operating income was $1.98 billion for the quarter, resulting in an operating margin of 22.9%."
    },
    "yoy_growth": {
      "value": 31.2,
      "unit": "%",
      "source": "

## Step 6 – Run Log (Append-Only On-Disk Audit)

In [196]:
import json
from pathlib import Path

log_path = Path('run_log.jsonl')
if log_path.exists():
    lines = log_path.read_text(encoding='utf-8').strip().split('\n')
    print(f'Run log: {len(lines)} entries in run_log.jsonl')
    print()
    print('Last 10 entries:')
    print('-' * 60)
    for line in lines[-10:]:
        entry = json.loads(line)
        print(f"[{entry.get('ts','')}] {entry.get('agent','')} | {entry.get('action','')}")
else:
    print('No run log found yet. Run the pipeline first.')

Run log: 3468 entries in run_log.jsonl

Last 10 entries:
------------------------------------------------------------
[2026-11-02T18:04:15.435509] Comparison Agent | TOOL_RESULT
[2026-11-02T18:04:15.436507] Comparison Agent | WRITE_MEMORY
[2026-11-02T18:04:15.437508] Comparison Agent | END
[2026-11-02T18:04:15.451400] Editor Manager | START
[2026-11-02T18:04:15.453402] Editor Manager | CALL_TOOL
[2026-11-02T18:04:15.454411] Editor Manager | TOOL_RESULT
[2026-11-02T18:04:15.455405] Editor Manager | CALL_TOOL
[2026-11-02T18:04:15.456399] Editor Manager | TOOL_RESULT
[2026-11-02T18:04:15.457399] Editor Manager | WRITE_MEMORY
[2026-11-02T18:04:15.457399] Editor Manager | END


## 🏁 Demo Complete

The FERAD multi-agent system successfully:
- ✅ Loaded and parsed the earnings report
- ✅ Extracted KPIs via regex + RAG (Extraction Agent)
- ✅ Generated sourced revenue commentary (Revenue Analyst)
- ✅ Identified and ranked risk signals (Risk Analyst)
- ✅ Assessed forward guidance (Outlook Analyst)
- ✅ Computed QoQ deltas (Comparison Agent)
- ✅ Detected analyst disagreements and assembled the final note (Editor Manager)
- ✅ All claims traceable to source sentences from the report

**Output:** `analyst_note.md` contains the one-page analyst note.
**Audit:** `run_log.jsonl` contains the append-only run log.